In [1]:
import requests
from urllib.parse import urlencode
import pandas as pd
import time
from IPython.display import display, clear_output
from datetime import timedelta

In [2]:
# ========== CONFIG ==========
RESOURCE_ID = "a63ab354-7e68-44c2-ad96-c6f920c30e85"
START = "2026-09-05T00:00:00.000Z"
END   = "2026-09-11T23:59:59.999Z"

PAGE_SIZE = 30000          # stay under the ~32k hard limit
SLEEP_SECONDS = 1         # respect ~2 req/min recommendation
BASE_URL = "https://api.neso.energy/api/3/action/datastore_search_sql"
# ============================

In [3]:
def fetch_page(offset: int, limit: int = PAGE_SIZE):
    sql = f'''
    SELECT COUNT(*) OVER () AS _count, *
    FROM "{RESOURCE_ID}"
    WHERE "deliveryStart" >= '{START}'
      AND "deliveryStart" <= '{END}'
    ORDER BY "_id" ASC
    LIMIT {limit} OFFSET {offset}
    '''
    params = {"sql": sql}
    url = f"{BASE_URL}?{urlencode(params)}"

    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    if not data.get("success"):
        raise RuntimeError(f"API error: {data}")

    return data["result"]

In [4]:
# ---- First page to discover total ----
print("Fetching first page to get total count...")
first = fetch_page(0)
total = int(first["records"][0]["_count"]) if first["records"] else 0
print(f"Total records in range: {total:,}")

all_records = first["records"]
print(f"Fetched {len(all_records):,} so far")

Fetching first page to get total count...
Total records in range: 52,211
Fetched 30,000 so far


In [5]:
# ---- Remaining pages ----
offset = PAGE_SIZE
while offset < total:
    print(f"Sleeping {SLEEP_SECONDS}s (rate-limit friendly)...")
    time.sleep(SLEEP_SECONDS)

    page = fetch_page(offset)
    records = page["records"]
    if not records:
        break

    all_records.extend(records)
    offset += PAGE_SIZE
    print(f"Fetched {len(all_records):,} / {total:,}")

print(f"\nDone. Total fetched: {len(all_records):,}")

Sleeping 1s (rate-limit friendly)...
Fetched 52,211 / 52,211

Done. Total fetched: 52,211


In [6]:
# Convert to DataFrame
df = pd.DataFrame(all_records)
df

,serviceType,technologyType,executedQuantity,_full_text,_count,auctionUnit,unitResultID,deliveryStart,deliveryEnd,registeredAuctionParticipant,clearingPrice,auctionProduct,_id,postCode
0,Quick Reserve,Batteries,39.0,"'-05':13,19 '-09':12,18 '-1':5 '00':15,16,22 '...",52211,DOLLB-1,2832#||#12#||#PQR#||#201564,2026-09-05T06:00:00,2026-09-05T06:30:00,Statkraft Markets GmbH,4.9,PQR,1177182,SS11 8UA
1,Slow Reserve,Gas Reciprocating Engines,5.0,"'-01':5 '-05':13,19 '-09':12,18 '0.82':10 '00'...",52211,BLGT-01,2832#||#3952#||#PSR#||#201600,2026-09-05T00:00:00,2026-09-05T00:30:00,Statkraft Markets GmbH,0.82,PSR,1177183,None
2,Quick Reserve,Biomass,20.0,"'-05':15,21 '-09':14,20 '-1':7 '00':17,18,24 '...",52211,PGPET-1,2832#||#3140#||#PQR#||#201568,2026-09-05T08:00:00,2026-09-05T08:30:00,PEAK GEN TOP CO LIMITED,9.37,PQR,1177184,PE2 7JB
3,Slow Reserve,Gas Reciprocating Engines,1.0,"'-01':5 '-05':13,19 '-09':12,18 '00':16,21,22 ...",52211,GEDL-01,2832#||#3986#||#PSR#||#201631,2026-09-05T15:30:00,2026-09-05T16:00:00,Statkraft Markets GmbH,3.47,PSR,1177185,None
4,Slow Reserve,Gas Reciprocating Engines,28.0,"'-01':5 '-05':13,19 '-09':12,18 '00':16,21,22 ...",52211,DNNG-01,2832#||#3993#||#PSR#||#201643,2026-09-05T21:30:00,2026-09-05T22:00:00,FORSA TRADING LIMITED,1.14,PSR,1177186,S252QE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52206,Quick Reserve,Batteries,40.0,"'-09':12,18 '-1':5 '-11':13,19 '00':15,16,22 '...",52211,THURB-1,2838#||#2312#||#NQR#||#203396,2026-09-11T10:00:00,2026-09-11T10:30:00,Statkraft Markets GmbH,11.09,NQR,1230060,RM18 8QR
52207,Quick Reserve,Batteries,43.0,"'-09':13,19 '-1':6 '-11':14,20 '0.0':11 '00':1...",52211,COVNB-1,2838#||#251#||#PQR#||#203500,2026-09-11T02:00:00,2026-09-11T02:30:00,EDF ENERGY CUSTOMERS LIMITED,0.0,PQR,1230061,CV2 1NQ
52208,Response,Batteries,35.0,"'-09':12,18 '-11':13,19 '00':15,16,21,22 '10':...",52211,AG-HLIM03,2838#||#10#||#DCL#||#203490,2026-09-11T14:00:00,2026-09-11T18:00:00,LIMEJUMP ENERGY LIMITED,6.45,DCL,1230062,SN16 9DX
52209,Response,Batteries,20.0,"'-09':12,18 '-1':6 '-11':13,19 '00':15,16,21,2...",52211,TYLNB-1,2838#||#1422#||#DCL#||#203487,2026-09-11T02:00:00,2026-09-11T06:00:00,EDF ENERGY CUSTOMERS LIMITED,2.85,DCL,1230063,IP8 4JL


In [7]:
# Optional: drop the helper columns if you don't need them
df = df.drop(columns=["_count", "_full_text"], errors="ignore")

print("\nShape:", df.shape)
print("\nColumns:", list(df.columns))
display(df.head())


Shape: (52211, 12)

Columns: ['serviceType', 'technologyType', 'executedQuantity', 'auctionUnit', 'unitResultID', 'deliveryStart', 'deliveryEnd', 'registeredAuctionParticipant', 'clearingPrice', 'auctionProduct', '_id', 'postCode']


,serviceType,technologyType,executedQuantity,auctionUnit,unitResultID,deliveryStart,deliveryEnd,registeredAuctionParticipant,clearingPrice,auctionProduct,_id,postCode
0,Quick Reserve,Batteries,39.0,DOLLB-1,2832#||#12#||#PQR#||#201564,2026-09-05T06:00:00,2026-09-05T06:30:00,Statkraft Markets GmbH,4.9,PQR,1177182,SS11 8UA
1,Slow Reserve,Gas Reciprocating Engines,5.0,BLGT-01,2832#||#3952#||#PSR#||#201600,2026-09-05T00:00:00,2026-09-05T00:30:00,Statkraft Markets GmbH,0.82,PSR,1177183,None
2,Quick Reserve,Biomass,20.0,PGPET-1,2832#||#3140#||#PQR#||#201568,2026-09-05T08:00:00,2026-09-05T08:30:00,PEAK GEN TOP CO LIMITED,9.37,PQR,1177184,PE2 7JB
3,Slow Reserve,Gas Reciprocating Engines,1.0,GEDL-01,2832#||#3986#||#PSR#||#201631,2026-09-05T15:30:00,2026-09-05T16:00:00,Statkraft Markets GmbH,3.47,PSR,1177185,None
4,Slow Reserve,Gas Reciprocating Engines,28.0,DNNG-01,2832#||#3993#||#PSR#||#201643,2026-09-05T21:30:00,2026-09-05T22:00:00,FORSA TRADING LIMITED,1.14,PSR,1177186,S252QE


In [8]:
# Convert to datetime
df["deliveryStart"] = pd.to_datetime(df["deliveryStart"])
df["deliveryEnd"]   = pd.to_datetime(df["deliveryEnd"])

# ---------- 3. Expand blocks > 30 minutes ----------
print("\nExpanding blocks longer than 30 minutes...")

expanded_rows = []

for _, row in df.iterrows():
    start = row["deliveryStart"]
    end   = row["deliveryEnd"]
    duration = end - start

    # Number of 30-minute slots
    n_slots = int(duration.total_seconds() // 1800)   # 1800 = 30*60

    if n_slots <= 1:
        # Keep original record as-is
        expanded_rows.append(row.to_dict())
    else:
        # Create one record for every 30-minute block
        for i in range(n_slots):
            new_row = row.to_dict()
            new_start = start + timedelta(minutes=30 * i)
            new_end   = new_start + timedelta(minutes=30)

            new_row["deliveryStart"] = new_start
            new_row["deliveryEnd"]   = new_end
            expanded_rows.append(new_row)

exploded = pd.DataFrame(expanded_rows)

print(f"Original records : {len(df):,}")
print(f"After expansion  : {len(exploded):,}")


Expanding blocks longer than 30 minutes...
Original records : 52,211
After expansion  : 118,025


In [9]:
# ---------- 4. Add Settlement Period (SP) ----------
# Formula you requested
exploded["settlementPeriod"] = (
    exploded["deliveryStart"].dt.hour * 2
    + (exploded["deliveryStart"].dt.minute // 30)
    + 1
)

In [10]:
# ---------- 5. Add deliveryDate (dd-mm-yyyy) ----------
exploded["deliveryDate"] = exploded["deliveryStart"].dt.strftime("%d-%m-%Y")

In [11]:
# Save
exploded.to_csv("eac_auction_results_2026-08-01_to_2026-08-12.csv", index=False)
print("\nSaved to eac_auction_results_2026-08-01_to_2026-08-12.csv")


Saved to eac_auction_results_2026-08-01_to_2026-08-12.csv
